In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pycirclize import Circos
import scipy.stats as stats

In [ ]:
excluded_samples=['ST002-1D_LUNG-pacbio-uwsc-group1']

In [ ]:
## read in mitoscope qc files
df_pb = pd.read_csv("../benchmark/pacbio/output/qc_summary.tsv", sep='\t')
df_pb[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = df_pb['Sample'].str.split('-', expand=True)

df_ont = pd.read_csv("../benchmark/ont/output/qc_summary.tsv", sep='\t')
df_ont[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = df_ont['Sample'].str.split('-', expand=True)

In [ ]:
## generate combined df with NUMTs called across all PB and ONT samples
comb_numt_df = pd.DataFrame()

for sample in df_pb['Sample']:
    file_path = f'../benchmark/pacbio/output/{sample}/numts/{sample}.numts.INS.tsv'
    numt_df = pd.read_csv(file_path, sep='\t')
    numt_df.columns = ['nchr', 'npos', 'mt_start', 'mt_end', 'DR', 'DV', 'AF']
    numt_df['Sample'] = sample
    comb_numt_df = pd.concat([comb_numt_df, numt_df])

for sample in df_ont['Sample']:
    file_path = f'../benchmark/ont/output/{sample}/numts/{sample}.numts.INS.tsv'
    numt_df = pd.read_csv(file_path, sep='\t')
    numt_df.columns = ['nchr', 'npos', 'mt_start', 'mt_end', 'DR', 'DV', 'AF']
    numt_df['Sample'] = sample
    comb_numt_df = pd.concat([comb_numt_df, numt_df])

comb_numt_df = comb_numt_df[comb_numt_df['nchr'] != 'chrM']
comb_numt_df['strand'] = np.where(comb_numt_df['mt_start'] < comb_numt_df['mt_end'], '+', '-')
comb_numt_df['mstart'] = np.where(comb_numt_df['mt_start'] < comb_numt_df['mt_end'], comb_numt_df['mt_start'], comb_numt_df['mt_end'])
comb_numt_df['mend'] = np.where(comb_numt_df['mt_start'] >= comb_numt_df['mt_end'], comb_numt_df['mt_start'], comb_numt_df['mt_end'])
comb_numt_df['length'] = comb_numt_df['mend'] - comb_numt_df['mstart'] + 1
comb_numt_df['id'] = comb_numt_df['nchr'] + "-" + comb_numt_df['npos'].astype(str) + "-" + comb_numt_df['mstart'].astype(str) + "-" + comb_numt_df['mend'].astype(str)
comb_numt_df = comb_numt_df[['id', 'nchr', 'npos', 'strand', 'length', 'mstart', 'mend', 'DR', 'DV', 'AF', 'Sample']]
comb_numt_df[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = comb_numt_df['Sample'].str.split('-', expand=True)
comb_numt_df = comb_numt_df[~(comb_numt_df['Sample'].isin(excluded_samples))]
comb_numt_df['Tissue'] = comb_numt_df['Tissue'].str.split('_', expand=True)[1].str.capitalize()

comb_numt_df


In [ ]:
## group together NUMTs that are the same 
tolerance = 100

df = comb_numt_df.sort_values(['nchr', 'npos', 'mstart', 'mend']).reset_index(drop=True)

numt_ids = [None] * len(df)
current_id = 1

for i in range(len(df)):
    if numt_ids[i] is not None:
        continue

    numt_ids[i] = f"NUMT_{current_id}"
    current = df.iloc[i]

    # grow cluster forward
    for j in range(i + 1, len(df)):
        row = df.iloc[j]

        if row['nchr'] != current['nchr']:
            break  # sorted by chr

        if abs(row['npos'] - df.loc[i, 'npos']) > tolerance:
            break  # window exceeded

        if (abs(row['mstart'] - current['mstart']) <= tolerance and abs(row['mend'] - current['mend']) <= tolerance):
            numt_ids[j] = f"NUMT_{current_id}"
    
    current_id += 1

df['NUMT_ID'] = numt_ids

## representive values for each NUMT group based on medians
rep_cols = {
    'npos':   'npos_rep',
    'length':'length_rep',
    'mstart':'mstart_rep',
    'mend':  'mend_rep'
}

for col, new_col in rep_cols.items():
    df[new_col] = (
        df.groupby('NUMT_ID')[col]
          .transform('median')
          .astype(int)
    )

df['NUMT_ID_long'] = df['nchr'] + "_" + df['npos_rep'].astype(str) + "_" + df['length_rep'].astype(str) + "_" + df['mstart_rep'].astype(str) + "_" + df['mend_rep'].astype(str)

comb_numt_df = df.copy()
comb_numt_df

In [ ]:
## get replicate support for each NUMT (replicates per sample and total replicates across all samples)
comb_numt_df['reps_sample_specific'] = (
    comb_numt_df
    .groupby(['NUMT_ID_long','Donor','Tissue'])['NUMT_ID_long']
    .transform('count')
)

comb_numt_df['reps_total'] = (
    comb_numt_df
    .groupby(['NUMT_ID_long'])['NUMT_ID_long']
    .transform('count')
)

def custom_join(series):
    return ','.join(series.astype(str))

comb_numt_df['reps_sample_specific_names'] = (
    comb_numt_df
    .groupby(['NUMT_ID_long','Donor','Tissue'])['Sample']
    .transform(custom_join)
)

comb_numt_df['reps_total_names'] = (
    comb_numt_df
    .groupby(['NUMT_ID_long'])['Sample']
    .transform(custom_join)
)

comb_numt_df['confidence'] = np.where(comb_numt_df['reps_total'] > 1, 'high', 'low')

comb_numt_df.to_csv(f"tables/numt_df_benchmarking_tissue_ALL.csv", index=False)
comb_numt_df

In [ ]:
## make separate dfs for high and low confidence NUMTs
high_conf_numts = comb_numt_df[comb_numt_df['confidence'] == 'high' ]
low_conf_numts =  comb_numt_df[comb_numt_df['confidence'] == 'low' ]

print(len(low_conf_numts))

In [ ]:
## make high confidence df with sample replicates collapsed
collapsed_numt_df = high_conf_numts.drop_duplicates(subset=['NUMT_ID_long', 'Donor', 'Tissue']).drop(
       columns=['id', 'npos', 'length', 'mstart', 'mend', 'DR', 'DV', 'AF', 'Sample', 'Seq_Tech', 'Center', 'Group'])

collapsed_numt_df['Donor_Tissue'] = collapsed_numt_df['Donor'] + "-" + collapsed_numt_df['Tissue']

collapsed_numt_df.to_csv(f"tables/numt_df_benchmarking_tissue_high_conf_collapsed.csv", index=False)
collapsed_numt_df


In [ ]:
## get NUMT counts per donor/tissue sample
counts = (
    collapsed_numt_df.groupby("Donor_Tissue", as_index=False)['NUMT_ID_long'].nunique()
    .rename(columns={"NUMT_ID_long":"count"})
)
counts[['Donor', 'Tissue']] = counts['Donor_Tissue'].str.split('-', expand=True)

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
fig, ax = plt.subplots(layout='constrained', figsize=(6, 4))

g = sns.barplot(
    data=counts,
    x="Donor_Tissue",
    order=['ST001-Liver', 'ST001-Lung', 'ST002-Lung', 'ST002-Colon', 'ST003-Brain', 'ST004-Brain'],
    y="count",
    hue="Donor",
    legend=False,
    ax=ax
)
plt.ylim(0,12)
ticks = ax.get_xticklabels()
labels = [t.get_text() for t in ticks]
donors  = [l.split("-")[0] for l in labels]  
tissues = ['\n\n' + l.split("-")[1] for l in labels]

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(donors, rotation=0)
ax.set_ylabel("NUMT Count")
ax.set_xlabel("")

sec = ax.secondary_xaxis(location=0)
sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
sec.tick_params('x', length=0)

for tick in sec.get_xticklabels():
    tick.set_fontweight("bold")
    #tick.set_fontsize(14)

midpoints = [0.5,2.5,3.5]
for x in midpoints:
    ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

plt.savefig(f"plots/fig6-numt_counts.pdf", dpi=300)
plt.show()
print(tissues)
print(donors)

In [ ]:
## look at individual NUMT counts per sample replicate / GCC
counts_ind = (
    comb_numt_df.groupby("Sample", as_index=False)['NUMT_ID_long'].nunique()
    .rename(columns={"NUMT_ID_long":"count"})
)
counts_ind[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = counts_ind['Sample'].str.split('-', expand=True)

sns.set_theme(style="ticks", context="talk", font_scale=0.75)

g = sns.catplot(
    data=counts_ind[counts_ind['Seq_Tech'] == 'pacbio'],
    x="Donor",
    y="count",
    col="Tissue",
    #col='Seq_Tech',    
    hue="Center",
    kind="bar",
    height=4,
    aspect=0.8,
    palette="muted",
    legend_out=False,
    sharey=True,
    sharex=False,  
)

sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5))
g.set_titles("{col_name}")
g.set_axis_labels("", "")
plt.tight_layout()
plt.show()

g = sns.catplot(
    data=counts_ind[counts_ind['Seq_Tech'] == 'ont'],
    x="Donor",
    y="count",
    col="Tissue",
    #col='Seq_Tech',    
    hue="Center",
    kind="bar",
    height=4,
    aspect=0.8,
    palette="muted",
    legend_out=False,
    sharey=True,
    sharex=False,  
)

sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5))
g.set_titles("{col_name}")
g.set_axis_labels("", "")
plt.tight_layout()
plt.show()

In [ ]:
## for unique NUMT calls, get total count + median/mean/std of length 
print(collapsed_numt_df[['NUMT_ID_long', 'length_rep']].drop_duplicates()['length_rep'].count())
print(collapsed_numt_df[['NUMT_ID_long', 'length_rep']].drop_duplicates()['length_rep'].median())
print(collapsed_numt_df[['NUMT_ID_long', 'length_rep']].drop_duplicates()['length_rep'].mean())
print(collapsed_numt_df[['NUMT_ID_long', 'length_rep']].drop_duplicates()['length_rep'].std())

In [ ]:
## visualize NUMT calls in each sample across GCCs/techs
comb_numt_df['Donor_Tissue'] = comb_numt_df['Donor'] + "-" + comb_numt_df['Tissue']

for d in comb_numt_df['Donor_Tissue'].unique():

    heatmap_data = comb_numt_df[comb_numt_df['Donor_Tissue'] == d].pivot(index=['Sample','Donor_Tissue'], columns='NUMT_ID_long', values='length_rep')

    plt.figure(figsize=(16, 6))
    sns.heatmap(heatmap_data, cmap="viridis", linewidths=1,annot=False, cbar_kws={'label': 'NUMT Length (bp)'})
    plt.title('')
    plt.xlabel('')
    plt.ylabel('Sample')
    plt.tight_layout()
    plt.show()


In [ ]:
## generate heatmap for unique NUMT presence across different samples (collapsed replicates)

# column order by chromosome,position
collapsed_numt_df['chr_num'] = (
    collapsed_numt_df['nchr']
    .str.replace('chr', '', regex=False)
    .replace({'X': 23, 'Y': 24, 'M': 25})
    .astype(int)
)

collapsed_numt_df['npos_rep'] = collapsed_numt_df['npos_rep'].astype(int)

column_order = (
    collapsed_numt_df[['NUMT_ID_long', 'chr_num', 'npos_rep']]
    .drop_duplicates()
    .sort_values(['chr_num', 'npos_rep'])
    ['NUMT_ID_long']
)

heatmap_data = collapsed_numt_df.sort_values('Donor_Tissue', ascending=True).pivot(index=['Donor_Tissue'], columns='NUMT_ID_long', values='length_rep')
heatmap_data = heatmap_data[column_order]
sort_order = ["ST004-Brain", "ST003-Brain", "ST002-Colon", "ST002-Lung", "ST001-Lung", "ST001-Liver"]
order_map = {t: i for i, t in enumerate(sort_order)}
heatmap_data = heatmap_data.sort_values(by='Donor_Tissue', key=lambda column: column.map(order_map))

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
plt.figure(figsize=(16, 8))

ax = sns.heatmap(
    heatmap_data,
    cmap="viridis",
    linewidths=0.5,
    annot=False,
    cbar_kws={'label': 'Length (bp)'}
)

#ax.invert_yaxis()
row_labels = heatmap_data.index.to_list()
donors = [x.split('-')[0] for x in row_labels]

boundary_positions = [
    i for i in range(1, len(donors))
    if donors[i] != donors[i - 1]
]

for y in boundary_positions:
    ax.hlines(y, *ax.get_xlim(), colors='black', linewidth=0.5,  linestyles='dashed')

for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1) 

plt.xlabel('')
plt.ylabel('')
plt.xticks(fontsize=11)
plt.tight_layout()
plt.savefig(f"plots/fig6-numt_heatmap.pdf", dpi=300)
plt.show()

In [ ]:
# count unique samples per NUMT_ID
collapsed_numt_df['freq'] = (
    collapsed_numt_df
    .groupby(['NUMT_ID_long'])['Donor_Tissue']
    .transform('count')
)

collapsed_numt_df

In [ ]:
## get list of unique NUMTs
smaht_unique_numts = collapsed_numt_df[['NUMT_ID_long', 'nchr', 'npos_rep', 'length_rep', 'mstart_rep', 'mend_rep']].drop_duplicates().reset_index()
smaht_unique_numts

In [ ]:
## get NUMT counts on each chrom
smaht_unique_numts['nchr'].value_counts()

In [ ]:
## circos plot for unique NUMTs

## base circos plot (outer track)
## ------------------------------------------------------------------ ##
mt_length = 16569 
scale_factor=200000
bed_file = "jn_resources/circos.bed"

# Initialize Circos with space between chromosomes
circos = Circos.initialize_from_bed(bed_file, space=2)
chr_names = [s.name for s in circos.sectors]

# Assign Colors
circos_colors = list(plt.cm.tab20(np.linspace(0, 1, 20)))  # 20 colors
circos_colors += list(plt.cm.tab20b(np.linspace(0, 1, 4)))  # 4 more colors
chr_name2color = {name: color for name, color in zip(chr_names, circos_colors)}

# Label chromosomes and draw outer track 
tick_interval = 2000
positions = [i * scale_factor for i in range(0, mt_length, tick_interval)]
labels = [f"{i//1000}K" for i in range(0, mt_length, tick_interval)]

for sector in circos.sectors:
    track = sector.add_track((92, 100))

    if sector.name in ["chrM", "chrMT"]:
       # sector.text('MT', size=12)
        track.xticks(
            positions,
            labels=labels,
            outer=True,
            label_orientation="vertical",
            label_size=12
        )
        track.axis(fc="#6baed6", lw=1)
    else:
        label = sector.name.replace("chr", "")
        sector.text(label, size=12)
        color = chr_name2color.get(sector.name, "gray")
        track.axis(fc=color, lw=1)
## ------------------------------------------------------------------ ##

smaht_unique_numts['mstart_rep'] = smaht_unique_numts['mstart_rep'].astype(int)
smaht_unique_numts['mend_rep'] = smaht_unique_numts['mend_rep'].astype(int)

# plot links
for _, row in smaht_unique_numts.iterrows():
    region_mt = ("chrMT", int(row["mstart_rep"]*scale_factor), int(row["mend_rep"]*scale_factor))
    region_nuc = (row["nchr"], int(row["npos_rep"]), int(row["npos_rep"] + 1))
    color = chr_name2color.get(row["nchr"], "blue")

    # norm = colors.Normalize(vmin=collapsed_numt_df["freq"].min(), vmax=collapsed_numt_df["freq"].max())
    # color = cm.viridis(norm(row["freq"]))

    circos.link(region_mt, region_nuc, color=color, alpha=0.8, lw=2)

circos.savefig(f"plots/fig6-numt_circos.pdf", dpi=300)
circos.plotfig()

In [ ]:
## generate combined df with NUMTs called across all PB and ONT samples (seqfirst)
seqfirst_pb_samples = pd.read_csv('../../seqfirst/pacbio/samples_for_merge_pb2.csv')
seqfirst_ont_samples = pd.read_csv('../../seqfirst/ont/samples_for_merge_ont.csv')

seqfirst_numt_df = pd.DataFrame()

for i in range(0, len(seqfirst_pb_samples)):
    row = seqfirst_pb_samples.iloc[i]
    sample = row['sample']
    file_path = f'../../seqfirst/pacbio/output/{sample}/numts/{sample}.numts.INS.tsv'
    numt_df = pd.read_csv(file_path, sep='\t')
    numt_df.columns = ['nchr', 'npos', 'mt_start', 'mt_end', 'DR', 'DV', 'AF']
    numt_df['Sample'] = sample
    numt_df['Family_ID_Member'] = row['family_id_member']
    numt_df['Seq_Tech'] = 'pacbio'
    seqfirst_numt_df = pd.concat([seqfirst_numt_df, numt_df])

for i in range(0, len(seqfirst_ont_samples)):
    row = seqfirst_ont_samples.iloc[i]
    sample = row['sample']
    if sample != '-':
        file_path = f'../../seqfirst/ont/output/{sample}/numts/{sample}.numts.INS.tsv'
        numt_df = pd.read_csv(file_path, sep='\t')
        numt_df.columns = ['nchr', 'npos', 'mt_start', 'mt_end', 'DR', 'DV', 'AF']
        numt_df['Sample'] = sample
        numt_df['Family_ID_Member'] = row['family_id_member']
        numt_df['Seq_Tech'] = 'ont'
        seqfirst_numt_df = pd.concat([seqfirst_numt_df, numt_df])

seqfirst_numt_df = seqfirst_numt_df[seqfirst_numt_df['nchr'] != 'chrM']
seqfirst_numt_df['strand'] = np.where(seqfirst_numt_df['mt_start'] < seqfirst_numt_df['mt_end'], '+', '-')
seqfirst_numt_df['mstart'] = np.where(seqfirst_numt_df['mt_start'] < seqfirst_numt_df['mt_end'], seqfirst_numt_df['mt_start'], seqfirst_numt_df['mt_end'])
seqfirst_numt_df['mend'] = np.where(seqfirst_numt_df['mt_start'] >= seqfirst_numt_df['mt_end'], seqfirst_numt_df['mt_start'], seqfirst_numt_df['mt_end'])
seqfirst_numt_df['length'] = seqfirst_numt_df['mend'] - seqfirst_numt_df['mstart'] + 1
seqfirst_numt_df['id'] = seqfirst_numt_df['nchr'] + "-" + seqfirst_numt_df['npos'].astype(str) + "-" + seqfirst_numt_df['mstart'].astype(str) + "-" + seqfirst_numt_df['mend'].astype(str)
seqfirst_numt_df = seqfirst_numt_df[['id', 'nchr', 'npos', 'strand', 'length', 'mstart', 'mend', 'Sample', 'Family_ID_Member', 'Seq_Tech']]

chromosomes = [f'chr{x}' for x in list(range(1, 22)) + ['X', 'Y']]
seqfirst_numt_df = seqfirst_numt_df[seqfirst_numt_df['nchr'].isin(chromosomes)]
seqfirst_numt_df

In [ ]:
## select seqfirst samples that have both PB and ONT data for use in NUMT analysis
seqfirst_family_id_df = pd.merge(seqfirst_pb_samples, seqfirst_ont_samples, on='family_id_member', how='outer')
seqfirst_family_id_df.columns = ['family_id_member', 'pb', 'ont']
seqfirst_family_id_df

only_1_seqtech = seqfirst_family_id_df[(seqfirst_family_id_df['ont'] == '-') | (seqfirst_family_id_df['pb'] == '-')]['family_id_member'].to_list()
only_1_seqtech

## skip samples that only have PB or only ONT (use only those with both)
seqfirst_numt_df = seqfirst_numt_df[~(seqfirst_numt_df['Family_ID_Member'].isin(only_1_seqtech))]
seqfirst_numt_df['Family_ID_Member'].nunique()

In [ ]:
## group together NUMTs that are the same (for seqfirst)
tolerance = 100

df = seqfirst_numt_df.sort_values(['nchr', 'npos', 'mstart', 'mend']).reset_index(drop=True)

numt_ids = [None] * len(df)
current_id = 1

for i in range(len(df)):
    if numt_ids[i] is not None:
        continue

    numt_ids[i] = f"NUMT_{current_id}"
    current = df.iloc[i]

    # grow cluster forward
    for j in range(i + 1, len(df)):
        row = df.iloc[j]

        if row['nchr'] != current['nchr']:
            break  # sorted by chr

        if abs(row['npos'] - current['npos']) > tolerance:
            break  # window exceeded

        if (abs(row['mstart'] - current['mstart']) <= tolerance and abs(row['mend'] - current['mend']) <= tolerance):
            numt_ids[j] = f"NUMT_{current_id}"

    current_id += 1

df['NUMT_ID'] = numt_ids

## representive values for each NUMT group based on medians
rep_cols = {
    'npos':   'npos_rep',
    'length':'length_rep',
    'mstart':'mstart_rep',
    'mend':  'mend_rep'
}

for col, new_col in rep_cols.items():
    df[new_col] = (
        df.groupby('NUMT_ID')[col]
          .transform('median')
          .astype(int)
    )

df['NUMT_ID_long'] = df['nchr'] + "_" + df['npos_rep'].astype(str) + "_" + df['length_rep'].astype(str) + "_" + df['mstart_rep'].astype(str) + "_" + df['mend_rep'].astype(str)

seqfirst_numt_df = df.copy()
seqfirst_numt_df

In [ ]:
## get replicate support for each NUMT (replicates per sample and total replicates across all samples)
seqfirst_numt_df['reps_sample_specific'] = (
    seqfirst_numt_df
    .groupby(['NUMT_ID_long', 'Family_ID_Member'])['NUMT_ID_long']
    .transform('count')
)

seqfirst_numt_df['rep_sample_specific_names'] = (
    seqfirst_numt_df
    .groupby(['NUMT_ID_long', 'Family_ID_Member'])['Seq_Tech']
    .transform(custom_join)
)

seqfirst_numt_df['reps_total'] = (
    seqfirst_numt_df
    .groupby(['NUMT_ID_long'])['NUMT_ID_long']
    .transform('count')
)

seqfirst_numt_df['rep_total_names'] = (
    seqfirst_numt_df
    .groupby(['NUMT_ID_long'])['Family_ID_Member']
    .transform(custom_join)
)

seqfirst_numt_df

In [ ]:
## singletons
seqfirst_numt_df[(seqfirst_numt_df['reps_sample_specific'] == 1) & (seqfirst_numt_df['reps_total'] == 1)].sort_values('reps_total')

In [ ]:
## seqfirst NUMTs with sample replicates (PB/ONT) collapsed
collapsed_seqfirst_numt_df = seqfirst_numt_df.drop_duplicates(subset=['Family_ID_Member', 'NUMT_ID_long'])
collapsed_seqfirst_numt_df

In [ ]:
## quick look at what percent of NUMT calls have support from both PB and ONT
seqfirst_numt_df.groupby('Seq_Tech')['reps_sample_specific'].value_counts()

# 141 + 313 + 2917 = 3317 total calls
# 2917 / 3317 = 88% with ont + pb support

In [ ]:
## calculate NUMT frequencies within seqfirst and add to df 
total_seqfirst_samples = collapsed_seqfirst_numt_df['Family_ID_Member'].nunique()
total_seqfirst_samples

collapsed_seqfirst_numt_df['seqfirst_sample_count'] = (
    collapsed_seqfirst_numt_df
    .groupby(['NUMT_ID_long'])['Family_ID_Member']
    .transform('count')
)
collapsed_seqfirst_numt_df['seqfirst_AF'] = collapsed_seqfirst_numt_df['seqfirst_sample_count'] / total_seqfirst_samples

# # seqfirst_freqs = collapsed_seqfirst_numt_df.groupby('NUMT_ID_long').agg(sample_count=('Family_ID_Member', 'count')).reset_index()
# # seqfirst_freqs['PAF'] = seqfirst_freqs['sample_count'] / total_seqfirst_samples

conditions = [
    (collapsed_seqfirst_numt_df['seqfirst_AF'] >= 0.01),
    (collapsed_seqfirst_numt_df['seqfirst_AF'] >= 0.001),
    (collapsed_seqfirst_numt_df['seqfirst_AF'] < 0.001)
]
choices = ['Common', 'Rare', 'Ultra-rare']
collapsed_seqfirst_numt_df['seqfirst_freq_group'] = np.select(conditions, choices, default='Unknown')
collapsed_seqfirst_numt_df['seqfirst_freq_group'].value_counts()
collapsed_seqfirst_numt_df


In [ ]:
## generate list of unique NUMTs
unique_seqfirst_numts = collapsed_seqfirst_numt_df[['NUMT_ID_long', 'nchr', 'npos_rep', 'length_rep', 'mstart_rep', 'mend_rep', 'seqfirst_sample_count', 'seqfirst_AF', 'seqfirst_freq_group']].drop_duplicates().reset_index(drop=True)
unique_seqfirst_numts.sort_values('seqfirst_sample_count')

In [ ]:
## read in 1637 NUMTs from short read catalog of 100K Genomes Project
gen_england_db = pd.read_csv('jn_resources/england_numt_db.csv')

conditions = [
    (gen_england_db['frequency(all populations)'] >= 0.01),
    (gen_england_db['frequency(all populations)'] >= 0.001),
    (gen_england_db['frequency(all populations)'] < 0.001)
]
choices = ['Common', 'Rare', 'Ultra-rare']
gen_england_db['Frequency_Group'] = np.select(conditions, choices, default='Unknown')
gen_england_db['Frequency_Group'].value_counts()

In [ ]:
# annotate seqfirst NUMTs with gen_england_db freqs
seqfirst_merged = unique_seqfirst_numts.merge(gen_england_db, left_on="nchr", right_on="chromosome1",  how="left", suffixes=("", "_db"))
tolerance = 100

seqfirst_merged["bp_ok"] = (
    ((abs(seqfirst_merged["nuclearGenome_breakpoint1"] - tolerance)) < seqfirst_merged["npos_rep"]) & ((abs(seqfirst_merged["nuclearGenome_breakpoint2"] + tolerance)) > seqfirst_merged["npos_rep"]))

seqfirst_merged['diff'] = abs(seqfirst_merged['npos_rep'] - seqfirst_merged['nuclearGenome_breakpoint1'])
seqfirst_merged[seqfirst_merged['bp_ok'] == True]['NUMT_ID_long'].nunique()

matched_numts = (
    seqfirst_merged
    .loc[seqfirst_merged["bp_ok"], "NUMT_ID_long"]
    .unique()
)

unique_seqfirst_numts["in_SR_catalog"] = (
    unique_seqfirst_numts["NUMT_ID_long"].isin(matched_numts)
)

#unique_seqfirst_numts[unique_seqfirst_numts['length_rep'] < 500]
## >80% of seqfirst NUMTs are <500bp

#unique_seqfirst_numts[unique_seqfirst_numts['in_SR_catalog']]
## 86 / 168 seqfirst NUMTs are in the SR catalog

seqfirst_w_england = seqfirst_merged[seqfirst_merged['bp_ok'] == True]
unique_seqfirst_numts = unique_seqfirst_numts.merge(seqfirst_w_england[['NUMT_ID_long', 'frequency(all populations)', 'Frequency_Group']], on='NUMT_ID_long', how='left')


unique_seqfirst_numts.to_csv('tables/seqfirst_unique_numts.csv', index=False)
unique_seqfirst_numts

In [ ]:
## plot seqfirst NUMT length and sample count distribution
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

#fig, axs = plt.subplots(2, 1, sharex=True)
fig, axs = plt.subplots(
    2, 1,
    figsize=(5, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 3]}
)
sns.histplot(unique_seqfirst_numts, x='length_rep', bins=20, alpha=0.8, ax=axs[0])
sns.scatterplot(unique_seqfirst_numts, x='length_rep', y='seqfirst_sample_count', hue='in_SR_catalog', s=60, alpha=0.9, ax=axs[1])

axs[0].set_yscale("log")
#axs[1].set_xscale("log")
axs[0].set_ylabel("Frequency")
axs[1].set_ylabel("Sample Count")
axs[1].set_xlabel("Length (bp)")

plt.savefig(f"plots/fig6-numt_seqfirst_distribution.pdf", dpi=300)
plt.show()

In [ ]:
# annotate benchmarking NUMTs with gen_england_db freqs
merged = smaht_unique_numts.merge(gen_england_db, left_on="nchr", right_on="chromosome1",  how="left", suffixes=("", "_db"))
tolerance = 100

merged["bp_ok"] = (
    ((abs(merged["nuclearGenome_breakpoint1"] - tolerance)) < merged["npos_rep"]) & ((abs(merged["nuclearGenome_breakpoint2"] + tolerance)) > merged["npos_rep"]))

matched_numts = (
    merged
    .loc[merged["bp_ok"], "NUMT_ID_long"]
    .unique()
)

smaht_unique_numts["in_SR_catalog"] = (
    smaht_unique_numts["NUMT_ID_long"].isin(matched_numts)
)

smaht_w_england = merged[merged['bp_ok'] == True]
smaht_unique_numts = smaht_unique_numts.merge(smaht_w_england[['NUMT_ID_long', 'frequency(all populations)', 'Frequency_Group']], on='NUMT_ID_long', how='left')
smaht_unique_numts

#smaht_unique_numts.to_csv('tables/numt_df_benchmarking_tissue_unique.csv', sep='\t', index=False)


In [ ]:
## plot benchmarking NUMTs with frequencies in gen end db 
freq_counts = smaht_w_england['Frequency_Group'].value_counts().reset_index()
freq_counts.loc[len(freq_counts)] = ['Absent', len(smaht_unique_numts[smaht_unique_numts['in_SR_catalog'] == False])]

sns.set_theme(style="ticks", context="talk", font_scale=0.9)
plt.figure(figsize=(5,4))
sns.barplot(freq_counts, x='Frequency_Group', y='count', hue='Frequency_Group', palette='Set2', width=0.8)
plt.xlabel('')
plt.xticks(rotation=45)
plt.ylabel('NUMT Count')

plt.savefig(f"plots/fig6-numts_SR_catalog.pdf", dpi=300)
plt.show()

In [ ]:
# annotate benchmarking NUMTs with seqfirst freqs
merged = smaht_unique_numts.merge(unique_seqfirst_numts, on="nchr",  how="left", suffixes=("", "_db"))
TOL = 100

merged["bp_ok"] = (
    (abs(merged["npos_rep"] - merged["npos_rep_db"]) < TOL ))

matched_numts = (
    merged
    .loc[merged["bp_ok"], "NUMT_ID_long"]
    .unique()
)

smaht_unique_numts["in_seqfirst"] = (
    smaht_unique_numts["NUMT_ID_long"].isin(matched_numts)
)

smaht_unique_numts = smaht_unique_numts.merge(unique_seqfirst_numts[['NUMT_ID_long', 'seqfirst_sample_count', 'seqfirst_AF', 'seqfirst_freq_group']],
      on='NUMT_ID_long', how='left')

smaht_unique_numts.to_csv('tables/numt_df_benchmarking_tissue_unique.csv', index=False)
smaht_unique_numts

In [ ]:
## circos plot for unique NUMTs (seqfirst)

## base circos plot (outer track)
## ------------------------------------------------------------------ ##
mt_length = 16569 
scale_factor=200000
bed_file = "jn_resources/circos.bed"

# Initialize Circos with space between chromosomes
circos = Circos.initialize_from_bed(bed_file, space=2)
chr_names = [s.name for s in circos.sectors]

# Assign Colors
circos_colors = list(plt.cm.tab20(np.linspace(0, 1, 20)))  # 20 colors
circos_colors += list(plt.cm.tab20b(np.linspace(0, 1, 4)))  # 4 more colors
chr_name2color = {name: color for name, color in zip(chr_names, circos_colors)}

# Label chromosomes and draw outer track
tick_interval = 2000
positions = [i * scale_factor for i in range(0, mt_length, tick_interval)]
labels = [f"{i//1000}K" for i in range(0, mt_length, tick_interval)]

for sector in circos.sectors:
    track = sector.add_track((92, 100))

    if sector.name in ["chrM", "chrMT"]:
       # sector.text('MT', size=12)
        track.xticks(
            positions,
            labels=labels,
            outer=True,
            label_orientation="vertical",
            label_size=12
        )
        track.axis(fc="#6baed6", lw=1)
    else:
        label = sector.name.replace("chr", "")
        sector.text(label, size=12)
        color = chr_name2color.get(sector.name, "gray")
        track.axis(fc=color, lw=1)
## ------------------------------------------------------------------ ##

unique_seqfirst_numts['mstart_rep'] = unique_seqfirst_numts['mstart_rep'].astype(int)
unique_seqfirst_numts['mend_rep'] = unique_seqfirst_numts['mend_rep'].astype(int)

# plot links
for _, row in unique_seqfirst_numts.iterrows():
    region_mt = ("chrMT", int(row["mstart_rep"]*scale_factor), int(row["mend_rep"]*scale_factor))
    region_nuc = (row["nchr"], int(row["npos_rep"]), int(row["npos_rep"] + 1))
    color = chr_name2color.get(row["nchr"], "blue")

    # norm = colors.Normalize(vmin=collapsed_numt_df["freq"].min(), vmax=collapsed_numt_df["freq"].max())
    # color = cm.viridis(norm(row["freq"]))

    circos.link(region_mt, region_nuc, color=color, alpha=0.25, lw=1)

circos.plotfig()

In [ ]:
## generate expected trinucleotide frequencies from hg38

# from collections import Counter
# from Bio import SeqIO
# import itertools

# # generate all possible 3-mers
# bases = ['A','C','G','T']
# kmers = [''.join(p) for p in itertools.product(bases, repeat=3)]

# counts = Counter()

# for record in SeqIO.parse("../../../../ref/GRCh38_no_alt_analysis_set.fasta","fasta"):
#     print(record.id)
#     if record.id in valid_chroms:
#         seq = str(record.seq).upper()
        
#         for kmer in kmers:
#             counts[kmer] += seq.count(kmer)

# tri_df = pd.DataFrame.from_dict(counts, orient='index').reset_index()
# tri_df.columns = ['trinuc', 'kmer_count']
# tri_df.to_csv('homology_analysis/hg38_chr1-22XY_trinucleotide_frequencies.csv', index=False)


In [ ]:
## generate expected trinucleotide frequencies from mito ref

# from collections import Counter
# from Bio import SeqIO
# import itertools

# # generate all possible 3-mers
# bases = ['A','C','G','T']
# kmers = [''.join(p) for p in itertools.product(bases, repeat=3)]

# counts = Counter()

# for record in SeqIO.parse("../../../../tools/mitoscope/resources/MT.fasta","fasta"):
#     seq = str(record.seq).upper()
    
#     for kmer in kmers:
#         counts[kmer] += seq.count(kmer)

# mito_tri_df = pd.DataFrame.from_dict(counts, orient='index').reset_index()
# mito_tri_df.columns = ['trinuc', 'kmer_count']
# mito_tri_df.to_csv('homology_analysis/mt_trinucleotide_frequencies.csv', index=False)


In [ ]:
## generate dfs with expected nuc/mt trinucleotide frequencies
def rev_comp(sequence):
    bases = {'A':'T', 'T': 'A', 'G':'C', 'C':'G'}
    
    new_seq = ''
    for nuc in sequence:
        new_seq = bases[nuc] + new_seq
    return new_seq

mito_tri_df = pd.read_csv('homology_analysis/mt_trinucleotide_frequencies.csv')
mito_tri_df['exp_prop'] = mito_tri_df['kmer_count'] / mito_tri_df['kmer_count'].sum()
mito_tri_df['trinuc_collapsed'] = np.where(mito_tri_df['trinuc'].str[1].isin(['A', 'G']), mito_tri_df['trinuc'].apply(rev_comp), mito_tri_df['trinuc'])
mito_tri_df = mito_tri_df.groupby(['trinuc_collapsed']).sum().reset_index()
mito_tri_df

nuc_tri_df = pd.read_csv('homology_analysis/hg38_chr1-22XY_trinucleotide_frequencies.csv')
nuc_tri_df['exp_prop'] = nuc_tri_df['kmer_count'] / nuc_tri_df['kmer_count'].sum()
nuc_tri_df['trinuc_collapsed'] = np.where(nuc_tri_df['trinuc'].str[1].isin(['A', 'G']), nuc_tri_df['trinuc'].apply(rev_comp), nuc_tri_df['trinuc'])
nuc_tri_df = nuc_tri_df.groupby(['trinuc_collapsed']).sum().reset_index()
nuc_tri_df

In [ ]:
## generate nuc and mito dfs with trinucleotides surrounding NUMT breakpoints in seqfirst samples
tri = pd.read_csv('tables/seqfirst_unique_numts.trinuc.csv', sep='\t')

nuc_col = [col for col in tri if col.startswith('nuc')]
mito_col = [col for col in tri if col.startswith('mito')]

nuc_tri = pd.melt(tri, id_vars=['id'], value_vars=nuc_col, var_name='side', value_name='bases')
nuc_props = nuc_tri['bases'].value_counts().reset_index()
nuc_props['obs_prop'] = nuc_props['count'] / nuc_props['count'].sum()
nuc_props['converted_bases'] = np.where(nuc_props['bases'].str[1].isin(['A', 'G']), nuc_props['bases'].apply(rev_comp), nuc_props['bases'])
nuc_props = nuc_props.groupby(['converted_bases']).sum().reset_index()
nuc_props = pd.merge(nuc_props, nuc_tri_df[['trinuc_collapsed', 'exp_prop']], how='left', left_on='converted_bases', right_on='trinuc_collapsed')
nuc_props['log2_ratio'] = np.log2(nuc_props['obs_prop'] / nuc_props['exp_prop'])
nuc_props['genome'] = 'nuc'
nuc_props_long = pd.melt(nuc_props, id_vars=['converted_bases'], value_vars=['obs_prop','exp_prop', 'log2_ratio'],var_name='group', value_name='value')

mito_tri = pd.melt(tri, id_vars=['id'], value_vars=mito_col, var_name='side', value_name='bases')
mito_tri = mito_tri[mito_tri['bases'] != '-']
mito_props = mito_tri['bases'].value_counts().reset_index()
mito_props['obs_prop'] = mito_props['count'] / mito_props['count'].sum()
mito_props['converted_bases'] = np.where(mito_props['bases'].str[1].isin(['A', 'G']), mito_props['bases'].apply(rev_comp), mito_props['bases'])
mito_props = mito_props.groupby(['converted_bases']).sum().reset_index()
mito_props = pd.merge(mito_props, mito_tri_df[['trinuc_collapsed', 'exp_prop']], how='left', left_on='converted_bases', right_on='trinuc_collapsed')
mito_props['log2_ratio'] = np.log2(mito_props['obs_prop'] / mito_props['exp_prop'])
mito_props['genome'] = 'mito'
mito_props_long = pd.melt(mito_props, id_vars=['converted_bases'], value_vars=['obs_prop','exp_prop', 'log2_ratio'],var_name='group', value_name='value')

comb_df = pd.concat([nuc_props, mito_props])

comb_df.to_csv(f"tables/seqfirst_numt_trinucleotide_distributions.csv", index=False)
comb_df

In [ ]:
## trinuc distribution plot 
sns.set_theme(style="ticks", context="talk", font_scale=0.8)
plt.figure(figsize=(12,4))

sns.barplot(
    data=comb_df.sort_values('converted_bases'),
    x='converted_bases',
    y='log2_ratio',
    hue='genome'
)

plt.axhline(0, color='black', linewidth=1)
plt.grid(axis='y',color = 'gray', linestyle = '--', linewidth = 0.5)
plt.xticks(rotation=90)
plt.ylabel('log2(Observed / Expected)')
plt.xlabel('Trinucleotides')
plt.savefig(f"plots/fig6-trinucleotides.pdf", dpi=300)
plt.show()


In [ ]:
## chi-square test for nuclear genome trinuc distribution
observed = nuc_props['count'].values
expected = nuc_props['exp_prop'].values * observed.sum()

# normalize proportions to sum to 1
exp_prop = nuc_props['exp_prop'].values
exp_prop = exp_prop / exp_prop.sum()
expected = exp_prop * observed.sum()

chi2, p = stats.chisquare(f_obs=observed, f_exp=expected)
print("Chi-square:", chi2)
print("p-value:", p)

In [ ]:
## chi-square test for mito genome trinuc distribution
observed = mito_props['count'].values
expected = mito_props['exp_prop'].values * observed.sum()

# normalize proportions to sum to 1
exp_prop = mito_props['exp_prop'].values
exp_prop = exp_prop / exp_prop.sum()
expected = exp_prop * observed.sum()

chi2, p = stats.chisquare(f_obs=observed, f_exp=expected)
print("Chi-square:", chi2)
print("p-value:", p)

In [ ]:
## binomial tests for nuclear genome trinucs
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
import numpy as np

pvals = []
obs = nuc_props['count'].values
N = nuc_props['count'].sum()
exp = nuc_props['exp_prop']
for k, p in zip(obs, exp):
    result = binomtest(k, N, p, alternative='two-sided')
    pvals.append(result.pvalue)
nuc_props['binom_p'] = pvals

nuc_props['binom_p_adj'] = multipletests(
    nuc_props['binom_p'],
    method='fdr_bh'
)[1]

nuc_props[nuc_props['binom_p_adj'] < 0.05]

In [ ]:
## binomial tests for mito genome trinucs
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
import numpy as np

pvals = []
obs = mito_props['count'].values
N = mito_props['count'].sum()
exp = mito_props['exp_prop']
for k, p in zip(obs, exp):
    result = binomtest(k, N, p, alternative='two-sided')
    pvals.append(result.pvalue)
mito_props['binom_p'] = pvals

mito_props['binom_p_adj'] = multipletests(
    mito_props['binom_p'],
    method='fdr_bh'
)[1]

mito_props[mito_props['binom_p_adj'] < 0.05]